# Build POI Dataset (ONE-TIME)

⚠️ This notebook builds `final_pois.gpkg` from raw OSM shapefiles.
⚠️ Run ONCE only.
⚠️ Do NOT rerun unless raw OSM data changes.

In [1]:
%pip install fiona


In [2]:
# Fiona installation handled above.


Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import os
import fiona

In [3]:
RAW_DIR = Path("/data/baliu/python_code/data/switzerland-251226-free copy")
OUT_DIR = Path("/data/baliu/python_code/data/version2/data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "final_pois_nob.gpkg"

In [ ]:
if OUT_GPKG.exists():
    raise RuntimeError(
        "final_pois_nob.gpkg already exists.\n"
        "Delete it manually if want to rebuild."
    )

In [ ]:
from notebook_utils.poi_build import read_poi_layer as _safe_read


In [6]:
# read pois 
def _read_poi_files():

    gdfs_nob = []

    # --- pofw ---
    p = _safe_read(RAW_DIR / "gis_osm_pofw_free_1.shp", "pofw")
    if p is not None:
        gdfs_nob.append(p)

    # --- transport ---
    p = _safe_read(RAW_DIR / "gis_osm_transport_free_1.shp", "transport")
    if p is not None:
        gdfs_nob.append(p)

    # --- natural (drop trees) ---
    p = _safe_read(RAW_DIR / "gis_osm_natural_free_1.shp", "natural")
    if p is not None and "fclass" in p.columns:
        p = p.loc[p["fclass"] != "tree"]
        gdfs_nob.append(p)

    # --- natural areas → beaches → centroid ---
    p = _safe_read(RAW_DIR / "gis_osm_natural_a_free_1.shp", "natural_a")
    if p is not None and "fclass" in p.columns:
        p = p.loc[p["fclass"].isin(["beach"])].reset_index(drop=True)
        p["geometry"] = (
            p.to_crs("EPSG:2056")
             .geometry.centroid
             .to_crs("EPSG:4326")
        )
        gdfs_nob.append(p)

    # --- POIs ---
    p = _safe_read(RAW_DIR / "gis_osm_pois_free_1.shp", "pois")
    if p is not None:
        gdfs_nob.append(p)

    # --- traffic points (parking only) ---
    p = _safe_read(RAW_DIR / "gis_osm_traffic_free_1.shp", "traffic")
    if p is not None and "fclass" in p.columns:
        p = p.loc[p["fclass"].isin([
            "parking",
            "parking_bicycle",
            "parking_underground",
            "parking_multistorey",
        ])].reset_index(drop=True)
        gdfs_nob.append(p)

    # --- traffic areas → parking → centroid ---
    p = _safe_read(RAW_DIR / "gis_osm_traffic_a_free_1.shp", "traffic_a")
    if p is not None and "fclass" in p.columns:
        p = p.loc[p["fclass"].isin([
            "parking",
            "parking_bicycle",
            "parking_underground",
            "parking_multistorey",
        ])].reset_index(drop=True)
        p["geometry"] = (
            p.to_crs("EPSG:2056")
             .geometry.centroid
             .to_crs("EPSG:4326")
        )
        gdfs_nob.append(p)

    # # --- buildings → centroid + code ---    
    # p = _safe_read(RAW_DIR / "gis_osm_buildings_a_free_1.shp", "buildings")
    # if p is not None:
    #     if "type" in p.columns:
    #         p["code"] = p.groupby("type").ngroup() + 1002
    #     p["geometry"] = (
    #         p.to_crs("EPSG:2056")
    #          .geometry.centroid
    #          .to_crs("EPSG:4326")
    #     )
    #     gdfs_nob.append(p)

    if not gdfs_nob:
        raise RuntimeError("No POI layers could be loaded.")

    all_pois_nob = pd.concat(gdfs_nob, ignore_index=True)
    print("✅ Total POIs loaded:", len(all_pois_nob))
    return all_pois_nob

In [ ]:
from notebook_utils.poi_build import assign_category


In [12]:
from notebook_utils.poi_build import build_final_pois

def preprocess():
    all_pois_nob = _read_poi_files()
    all_pois_nob = build_final_pois(all_pois_nob)
    out_path = Path("/data/baliu/python_code/data/version2/data/final_pois_nob.gpkg")
    all_pois_nob.to_file(out_path, driver="GPKG")
    print("final_pois_nob.gpkg written:", len(all_pois_nob))
    print(all_pois_nob.head())


In [13]:
preprocess()


Reading pofw: gis_osm_pofw_free_1.shp
Reading transport: gis_osm_transport_free_1.shp
Reading natural: gis_osm_natural_free_1.shp
Reading natural_a: gis_osm_natural_a_free_1.shp
Reading pois: gis_osm_pois_free_1.shp
Reading traffic: gis_osm_traffic_free_1.shp
Reading traffic_a: gis_osm_traffic_a_free_1.shp
✅ Total POIs loaded: 534603
✅ final_pois_nob.gpkg written: 534603
   id  code                           name                         geometry  \
0   0  3100           Temple de Saint-Jean  POINT (2498816.948 1117839.539)   
1   1  3100  Kapelle Oberes Heiliges Kreuz  POINT (2691857.357 1192677.631)   
2   2  3102            Kirche St. Johannes  POINT (2599659.594 1202970.041)   
3   3  3102            Paroisse catholique  POINT (2501879.911 1126418.086)   
4   4  3300      Mosquée du Petit-Saconnex  POINT (2498411.634 1119984.893)   

  category  
0    Civic  
1    Civic  
2    Civic  
3    Civic  
4    Civic  
